In [ ]:
!pip install -q arviz

In [ ]:
!pip install -q pyspark

In [ ]:
!apt-get update

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

In [ ]:
!update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java

In [ ]:
!java -version

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.getenv("JAVA_HOME")

In [ ]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az
import scipy.stats as stats
import pymc3 as pm
import tensorflow as tf
import tensorflow_probability as tfp
import ipywidgets as widgets
from ipywidgets import interact

tfd = tfp.distributions
tfb = tfp.bijectors

import pyspark
from pyspark.sql import SparkSession
import pyspark.sql.functions as F


%matplotlib inline

In [ ]:
sc = SparkSession.builder.appName('LRAC-6330').getOrCreate()

#**LRAC-6330 - Researching ways to parallelize A/B Testing code in Spark**<hr span=30%></hr> 

**Description**

Research the possibility of run the AB Testing code using pymc3 in parallel with Spark. The key to do this is to run multiple Markov chains in parallel to accelerate convergence and to build a more robust model. Other options are to use Tensorflow Probability, but it requrires more fine tuning and low level aspects of the lib and more deep knowledge of the math involved.<br><br>

This notebook contains the models descripted in the LRAC-6330 ticket and it's sub-tasks.<br><br><br>




##1. Input Data
---
This is the test data as described in the [repo]

In [ ]:
dataset = pd.DataFrame(data={
		'trials':[3025,2895,3200,3523,2065,1645],
		'successes':[2235,2132,1789,3002,1245,1135],
		'days':[1,2,1,2,1,2]
},
index=['Control','Control','Variant A','Variant A','Variant B','Variant B']
)

dataset

In [ ]:
groups = dataset.groupby(dataset.index).agg({'trials':'sum','successes':'sum'})
groups

In [ ]:
probs = groups.assign(rate = lambda df: df['successes']/df['trials'])['rate']
probs

#**LRAC-6567** - Implement Bayesian Multi-Armed Bandits algorithms in Pyspark/Tensorflow

##**Description**

Bayesiam Multi-Armed Bandits have been used as an alternative for frequentist and bayesian A/B Testing. The aim of this ticket is to implement this algorithm using Tensorflow Probability to see if it'll be viable to research in the future.<br><br>


Unlike standard machine learning tools, bandit algorithms aren't simply black-box functions you can call to process the data you have lying around --- bandit algorithms have to actively select which data you should acquire and analyze that data in real-time. Indeed, bandit algorithms exemplify two types of learning that are not present in standard ML examples: *active learning*, which refers to algorithms that actively select which data they should receive; and *online learning*, which refers to algorithms that analyze data in real-time and provide results on the fly.<br><br>

#**Simulating the Arms of a Bandit Problem**

In order to reasonably simulate what might happen if we were to deploy an epsilon-Greedy algorithm in production, we need to set up some hypothetical arms. For this, we're going to focus on a very simple type of simulated arm that's easy to implement correctly. This hypothetical arm will let us simulate settings like:<br>

* *Optimizing click-throuhg rates for ads:* Every time we show someone an ad, we'll imagine that there's a fixed probability that they'll click on the ad. The bandit algorithm will then estimate this probability and try to decide on a strategy for showing ads that maximizes the click-through rate.
* *Conversion rates for new users:* Every time a new visitor comes to our site who isn't already a registered user, we'll imagine that there's a fixed probability that they'll register as a user after seeing the landing page. We'll then estimate this probability and try to decide on a strategy for maximizing our conversion rate.


# **Implementing the epsilon-Greedy algorithm**

Fields of the class:

**`epsilon`**<p></p>
>Floating point number that tells the frequency with which we should explore one of the available arms. Setting *epsilon* to 0.1 will allow the algorithm to explore the available arms on 10% of the pulls.

**`counts`**<p></p>
> A vector of integers of length **N** that tells us how many times we've played each of the **N** arms available to us in the current bandit problem. If there are 2 arms, Arm 1 and Arm 2, which have both been played twice, then we'll set *counts = [2, 2]*.

**`values`**<p></p>
>A vector of floating point numbers that defines the average amount of reward we've gotten when playing each of the **N** arms available to us. If Arm 1 gave us 1 unit of reward on one play and 0 on another play, while Arm 2 gave us 0 units of reward on both plays, then we'll set *values = [0.5, 0.0]*.

In [ ]:
class EpsilonGreedy:

	def __init__(self, epsilon, counts, values):
		self.epsilon = epsilon
		self.counts = counts
		self.values = values

		return

	def initialize(self, n_arms):

		self.counts = [0 for col in range(n_arms)]
		self.values = [0.0 for col in range(n_arms)]

		return

	def select_arm(self):
		if random.random() > self.epsilon:
			return np.argmax(self.values)
		else:
			return random.randrange(len(self.values))
	
	def update(self, chosen_arm, reward):
		self.counts[chosen_arm] = self.counts[chosen_arm] + 1
		n = self.counts[chosen_arm]

		value = self.values[chosen_arm]
		new_value = ((n - 1) / float(n)) * value + (1 / float(n)) * reward
		self.values[chosen_arm] = new_value

		return


## **Testing Framework**

Description: TODO

In [ ]:
class BernoulliArm:
	def __init__(self, p):
		self.p = p

	def draw(self):
		if random.random() > self.p:
			return 0.0
		else:
			return 1.0

In [ ]:
def test_bandits_algorithms(algo, arms, num_sims, horizon):
	
	chosen_arms = [0.0 for i in range(num_sims * horizon)]
	rewards = [0.0 for i in range(num_sims * horizon)]
	cumulative_rewards = [0.0 for i in range(num_sims * horizon)]
	sim_nums = [0.0 for i in range(num_sims * horizon)]
	times = [0.0 for i in range(num_sims * horizon)]

	for sim in range(num_sims):
		sim = sim + 1
		algo.initialize(len(arms))

		for t in range(horizon):
			t = t + 1
			index = (sim - 1) * horizon + t - 1
			sim_nums[index] = sim
			times[index] = t

			chosen_arm = algo.select_arm()
			chosen_arms[index] = chosen_arm
			reward = arms[chosen_arms[index]].draw()
			rewards[index] = reward

			if t == 1:
				cumulative_rewards[index] = reward 
			else:
				cumulative_rewards[index] = cumulative_rewards[index - 1] + reward
			algo.update(chosen_arm, reward)
		
		return [sim_nums, times, chosen_arms, rewards, cumulative_rewards]

In [ ]:
random.seed(42)
means = [0.1, 0.1, 0.1, 0.1, 0.9]
n_arms = len(means)
random.shuffle(means)
arms = [BernoulliArm(mean) for mean in means]
print('Best arm is ' + str(np.argmax(means)))

In [ ]:
np.argmax(means), means, t(means)

In [ ]:
final_result = []
num_sims = 5000
horizon = 250

for epsilon in [0.1, 0.2, 0.3, 0.4, 0.5]:
	algo = EpsilonGreedy(epsilon, [], [])
	algo.initialize(n_arms)
	results = test_bandits_algorithms(algo, arms, num_sims, horizon)
	for i in range(len(results[0])):
		final_result.append(
				{'epsilon': str(epsilon),
				 'data':[str(results[j][i]) for j in range(len(results))]
				 }
		)

In [ ]:
df_results = pd.DataFrame(final_result)

In [ ]:
df_results.head()

In [ ]:
df_results.describe()